# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 2c817f7a-39ae-426d-88b4-c8a61b… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ 1a3b8779-b9a0-40c5-9863-20bdc5… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-04 ┆ claude-hai ┆ 559.0      ┆ 1125.9    ┆ 15997428  ┆ 7956647   ┆ 0.049224  ┆ 44.62453  │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

### Tự đối chiếu các điều kiện Gold (notebook gốc chưa assert hết)

In [6]:
import os  # noqa: E402

print("Vị trí 3 bảng trên storage:")
for _name, _p in (("Bronze", BRONZE), ("Silver", SILVER), ("Gold", GOLD)):
    _dt = DeltaTable(_p)
    _parquet = [f for f in Path(_p).rglob("*.parquet") if "_delta_log" not in f.parts]
    print(f"  {_name:<6} {os.path.relpath(_p)}  version={_dt.version()}  files={len(_parquet)}"
          f"  rows={_dt.to_pyarrow_table().num_rows:,}")

_tz = con.sql("SELECT current_setting('TimeZone')").fetchone()[0]
print(f"\nDuckDB session TimeZone: {_tz}")
print("Silver rows per date (CAST(ts AS DATE) theo TimeZone của session):")
for _d, _n in con.sql("SELECT date, count(*) AS n FROM silver GROUP BY 1 ORDER BY 1").fetchall():
    print(f"  {_d}  {_n:>6,}")
print("Silver status values:", con.sql("SELECT status, count(*) FROM silver GROUP BY 1 ORDER BY 1").fetchall())

print("\nGold đầy đủ (date × model):")
with pl.Config(tbl_rows=-1, tbl_width_chars=160, fmt_str_lengths=20):
    print(gold_df.sort("date", "model").select(
        "date", "model", "p50_latency_ms", "p95_latency_ms",
        pl.col("error_rate").round(4), pl.col("cost_usd").round(2),
    ))

_per_model_dates = gold_df.group_by("model").agg(pl.col("date").n_unique().alias("dates"))
gold_checks = {
    "Silver < Bronze": silver_n < bronze_n,
    "≥ 7 dates": n_dates >= 7,
    "3 models": n_models == 3,
    "every model has ≥ 7 dates": bool((_per_model_dates["dates"] >= 7).all()),
    "rows = dates × models": gold_df.height == n_dates * n_models,
    "p50 ≤ p95 on every row": bool((gold_df["p50_latency_ms"] <= gold_df["p95_latency_ms"]).all()),
    "cost_usd > 0 on every row": bool((gold_df["cost_usd"] > 0).all()),
    "0 ≤ error_rate ≤ 1 on every row": bool(gold_df["error_rate"].is_between(0, 1).all()),
}
for k, v in gold_checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(gold_df.select(
    pl.col("p50_latency_ms").min().alias("min_p50"), pl.col("p95_latency_ms").max().alias("max_p95"),
    pl.col("cost_usd").min().alias("min_cost"), pl.col("cost_usd").sum().alias("total_cost"),
    pl.col("error_rate").min().alias("min_err"), pl.col("error_rate").max().alias("max_err"),
))
assert all(gold_checks.values()), "Gold incomplete — see FAIL rows above"

Vị trí 3 bảng trên storage:
  Bronze ../_lakehouse/bronze/llm_calls_raw  version=0  files=1  rows=200,000
  Silver ../_lakehouse/silver/llm_calls  version=0  files=8  rows=190,052
  Gold   ../_lakehouse/gold/llm_daily_metrics  version=1  files=16  rows=24

DuckDB session TimeZone: Asia/Ho_Chi_Minh
Silver rows per date (CAST(ts AS DATE) theo TimeZone của session):
  2026-04-01  19,271
  2026-04-02  27,199
  2026-04-03  27,096
  2026-04-04  27,176
  2026-04-05  27,131
  2026-04-06  27,152
  2026-04-07  27,112
  2026-04-08   7,915
Silver status values: [('error', 3757), ('ok', 180612), ('rate_limited', 5683)]

Gold đầy đủ (date × model):
shape: (24, 6)
┌────────────┬───────────────────┬────────────────┬────────────────┬────────────┬──────────┐
│ date       ┆ model             ┆ p50_latency_ms ┆ p95_latency_ms ┆ error_rate ┆ cost_usd │
│ ---        ┆ ---               ┆ ---            ┆ ---            ┆ ---        ┆ ---      │
│ date       ┆ str               ┆ f64            ┆ f64        

## 📝 Kết quả & giải thích (NB4)

| Tầng | Vị trí | Số dòng |
|---|---|---|
| Bronze | `_lakehouse/bronze/llm_calls_raw` (1 file) | **200.000** |
| Silver | `_lakehouse/silver/llm_calls` (partition theo `date`, 8 file) | **190.052**, dedup bỏ **9.948** |
| Gold | `_lakehouse/gold/llm_daily_metrics` (partition `date`, z-order `model`) | **24** = 8 ngày × 3 model |

Đối chiếu từng điều kiện Gold (cell kiểm tra tự thêm ở cuối, tất cả PASS):
- Silver < Bronze.
- 8 ngày ≥ 7, và mỗi model đều có ≥ 7 ngày. 3 model: haiku-4-5, sonnet-4-6, opus-4-7.
- p50 ≤ p95 trên mọi dòng. p50 nhỏ nhất 558 ms (haiku), p95 lớn nhất 6.058 ms (opus).
- `cost_usd` > 0 trên mọi dòng: nhỏ nhất $13,50, tổng $4.754,78 theo giá minh họa của lab.
- `error_rate` nằm trong [0,1] trên mọi dòng: khoảng 0,042–0,062.
- Thứ tự hợp lý: opus chậm nhất và đắt nhất mỗi token; sonnet có tổng chi phí cao nhất vì lưu lượng lớn nhất.

**Dedup ở Silver giải quyết vấn đề nào?**
- Bronze có 9.948 bản ghi trùng `request_id`, do generator mô phỏng retry hoặc at-least-once delivery của pipeline log.
- Nếu không dedup, mỗi request bị đếm nhiều lần, làm phồng token, chi phí, số lỗi và lệch percentile latency.
- Silver giữ đúng 1 dòng mỗi `request_id` (`ROW_NUMBER() … rn = 1`), parse JSON thành cột có kiểu, và bỏ bản ghi không có `model`.

**Vì sao dashboard đọc Gold?**
- Gold đã tổng hợp sẵn theo (ngày, model): 24 dòng so với 190K dòng. Dashboard đọc nhanh, rẻ và ổn định.
- Định nghĩa metric (p50/p95, cách tính lỗi, bảng giá) nằm ở một chỗ, nên mọi dashboard cho cùng một con số.
- Dashboard không phải tự parse JSON hay dedup lại, và quyền truy cập có thể giới hạn ở dữ liệu tổng hợp, không chạm `user_id` hay raw payload.

**Cách tính error rate và chi phí có phù hợp với dữ liệu đầu vào không?** Đúng về công thức, nhưng có ba điểm cần lưu ý, tôi đã kiểm tra trên dữ liệu thật:
1. **Múi giờ.**
   - Bronze là 7 ngày UTC (2026-04-01 00:00 → 04-07 23:59 UTC). Nhưng `CAST(ts AS DATE)` dùng TimeZone của session DuckDB, trên máy này là `Asia/Ho_Chi_Minh` (UTC+7).
   - Vì vậy Gold có **8** ngày: 04-01 chỉ có 17 giờ (19.271 dòng), 04-08 chỉ có 7 giờ (7.915 dòng), các ngày giữa khoảng 27.100 dòng.
   - Ngưỡng ≥ 7 ngày vẫn đạt, nhưng hai ngày đầu/cuối là ngày lẻ, không so sánh trực tiếp được. Kết quả cũng phụ thuộc máy chạy.
   - Cách sửa đúng là đặt `SET TimeZone='UTC'` hoặc tính `date` theo UTC ngay ở Silver.
2. **Error rate.**
   - `status <> 'ok'` gộp cả `error` (3.757) và `rate_limited` (5.683).
   - Nếu dashboard phục vụ SLO của model thì nên tách `rate_limited`, vì đó là lỗi quota/capacity chứ không phải lỗi model.
   - Silver không có `status` NULL, nên phép `<>` không bỏ sót dòng nào.
3. **Chi phí.**
   - Chi phí = token × giá/1M token theo bảng giá minh họa, không phải giá thật.
   - Công thức tính cả token của request lỗi. Điều này hợp lý nếu nhà cung cấp vẫn tính phí, nhưng cần xác nhận với hóa đơn thực tế.